# Seeing the arrow of time through a blurry lens

A Hamiltonian flow never changes the fine-grained (Gibbs) entropy of an ensemble. An observer who looks through a Gaussian blur of width $\sigma$ sees the **coarse-grained** density $\rho_C = \rho * \mathcal N(0,\sigma^2 I)$, whose entropy $S_C$ can go up **or down**.

If the blur grows at a rate $r = \dot\sigma/\sigma$, then
$$\frac{dS_C}{dt} = \sigma^2\,\mathrm{tr}F\,(r - R),$$
where $F$ is the Fisher matrix of $\rho_C$ and $R$ is a *critical rate* that depends on the state. The coarse-grained entropy grows exactly when the blur outpaces $R$.

* For linear flows, $R$ never exceeds the largest stretching rate $\kappa$ of the flow [1].
* For nonlinear flows, $R \le \kappa$ is proved for all Gaussian states, and a finite resolution leaves a **margin proportional to $\sigma$**: for the pendulum the best Gaussian state reaches only $R^* = 1 - \sigma + \tfrac78\sigma^2 - \dots$ [2].

This notebook shows that margin on a real ensemble. We put pendulums on the unstable top, in the best Gaussian state for $\sigma_0 = 0.05$, and watch them through three lenses:

| lens | blur rate $r$ | expected |
|---|---|---|
| fixed | $0$ | entropy falls: the arrow of time runs backwards |
| slow | $0.90$ | below $R \approx 0.952$: entropy dips |
| just enough | $0.96$ | above $R \approx 0.952$ but below $\kappa = 1$: entropy never falls |

Everything below runs on a CPU in about a minute. **Runtime → Run all.**

[1] T. Namba, *Sharp criterion for the coarse-grained arrow of time in Hamiltonian dynamics*, [doi:10.5281/zenodo.23258135](https://doi.org/10.5281/zenodo.23258135)
[2] T. Namba, *A resolution gap for the coarse-grained arrow of time in nonlinear Hamiltonian flows*, v1.1, [doi:10.5281/zenodo.23320603](https://doi.org/10.5281/zenodo.23320603) · code: [github.com/neginyan/resolution-gap-arrow-of-time](https://github.com/neginyan/resolution-gap-arrow-of-time)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.ndimage import gaussian_filter
from IPython.display import HTML
from base64 import b64encode

sigma0 = 0.05          # initial blur
rates  = [0.0, 0.90, 0.96]   # try your own values here
T, dt  = 3.5, 0.005    # time span and RK4 step of the pendulum flow

## 1. The ensemble: the best Gaussian state of the pendulum

$H = p^2/2 - \cos q$. At the unstable top $(q,p) = (\pi, 0)$ the flow stretches along $v = (1,1)/\sqrt2$ and contracts along $w = (1,-1)/\sqrt2$, both at rate $\kappa = 1$.

The Gaussian state with the largest critical rate is (Ref. [2], Sec. IV) a segment of zero width through the top, lying along the contracting direction $w$, with standard deviation $\approx 2\sqrt{\sigma}$ along its length and tilted towards $-v$ by $\approx\sigma/4$. Being sharp across the stretching direction is what makes it look like it *un-mixes*.

We represent it by 6001 weighted points on that segment and move every point with the exact pendulum flow.

In [ ]:
v = np.array([1, 1]) / np.sqrt(2); w = np.array([1, -1]) / np.sqrt(2)
tilt = sigma0 / 4
direction = np.cos(tilt) * w - np.sin(tilt) * v
u = np.linspace(-6, 6, 6001)
weights = np.exp(-u**2 / 2); weights /= weights.sum()
X0 = np.array([np.pi, 0.0]) + np.outer(u * 2 * np.sqrt(sigma0), direction)

def flow(P):                      # dq/dt = p, dp/dt = -sin q
    return np.stack([P[:, 1], -np.sin(P[:, 0])], 1)

def rk4(P, h):
    k1 = flow(P); k2 = flow(P + h/2*k1); k3 = flow(P + h/2*k2); k4 = flow(P + h*k3)
    return P + h/6*(k1 + 2*k2 + 2*k3 + k4)

P = X0.copy(); times = [0.0]; states = [P.copy()]
for k in range(int(round(T / dt))):
    P = rk4(P, dt)
    if (k + 1) % 10 == 0:
        times.append(round((k + 1) * dt, 6)); states.append(P.copy())
times = np.array(times); states = np.array(states)
print(len(times), 'frames, from t = 0 to', times[-1])

## 2. Coarse-grained entropy

The coarse-grained density of the weighted points is the exact sum $\rho_C(x) = \sum_i w_i\,\mathcal N(x; X_i, \sigma^2 I)$. We evaluate it on a grid with spacing $\sigma/8$ by depositing the points with cloud-in-cell weights and convolving with the Gaussian in Fourier space (dividing out the cloud-in-cell smoothing), and integrate $-\rho_C\ln\rho_C$.

As an independent check, `entropy_exact` sums the Gaussian kernels point by point.

In [ ]:
def entropy_fft(P, sig, per_sigma=8, pad=8.0):
    dx = sig / per_sigma; lo = P.min(0) - pad*sig; hi = P.max(0) + pad*sig
    nx = int(np.ceil((hi[0]-lo[0]) / dx)) + 2; ny = int(np.ceil((hi[1]-lo[1]) / dx)) + 2
    g = np.zeros((nx, ny)); fx = (P[:, 0]-lo[0]) / dx; fy = (P[:, 1]-lo[1]) / dx
    ix = np.floor(fx).astype(int); iy = np.floor(fy).astype(int); ax = fx - ix; ay = fy - iy
    for a, b, c in [(0, 0, (1-ax)*(1-ay)), (1, 0, ax*(1-ay)), (0, 1, (1-ax)*ay), (1, 1, ax*ay)]:
        np.add.at(g, (ix+a, iy+b), weights * c)
    kx = 2*np.pi*np.fft.fftfreq(nx, dx); ky = 2*np.pi*np.fft.rfftfreq(ny, dx)
    G = np.fft.rfft2(g) * np.exp(-0.5*sig**2*(kx[:, None]**2 + ky[None, :]**2))
    G /= (np.sinc(kx*dx/(2*np.pi))**2)[:, None] * (np.sinc(ky*dx/(2*np.pi))**2)[None, :]
    rho = np.fft.irfft2(G, s=(nx, ny)) / dx**2
    m = rho > 1e-12 * rho.max()
    return -np.sum(rho[m] * np.log(rho[m])) * dx * dx

def entropy_exact(P, sig, per_sigma=8):
    dx = sig / per_sigma; lo = P.min(0) - 8*sig; hi = P.max(0) + 8*sig
    nx = int(np.ceil((hi[0]-lo[0]) / dx)) + 1; ny = int(np.ceil((hi[1]-lo[1]) / dx)) + 1
    rho = np.zeros((nx, ny)); Rk = int(np.ceil(7*sig/dx)); off = np.arange(-Rk, Rk+1)
    ix0 = np.round((P[:, 0]-lo[0]) / dx).astype(int); iy0 = np.round((P[:, 1]-lo[1]) / dx).astype(int)
    for s in range(0, len(P), 500):
        sl = slice(s, s+500); IX = ix0[sl, None] + off; IY = iy0[sl, None] + off
        ex = np.exp(-0.5*((lo[0] + IX*dx - P[sl, 0, None]) / sig)**2)
        ey = np.exp(-0.5*((lo[1] + IY*dx - P[sl, 1, None]) / sig)**2)
        K = weights[sl, None, None] * ex[:, :, None] * ey[:, None, :] / (2*np.pi*sig*sig)
        np.add.at(rho, (np.broadcast_to(IX[:, :, None], K.shape).ravel(),
                        np.broadcast_to(IY[:, None, :], K.shape).ravel()), K.ravel())
    m = rho > 1e-300
    return -np.sum(rho[m] * np.log(rho[m])) * dx * dx

S = {r: np.array([entropy_fft(Pk, sigma0*np.exp(r*t)) for t, Pk in zip(times, states)]) for r in rates}
for r in rates:
    d = np.diff(S[r])
    print(f'r = {r:4.2f}:  steps where S_C falls: {(d < 0).sum():3d}   lowest S_C - S_C(0) = {S[r].min() - S[r][0]:+.4f}')

## 3. The critical rate of the initial state

By the formula above, $R = -\Sigma/(d S_C/d\ln\sigma)$, where $\Sigma$ is $dS_C/dt$ at fixed blur. We measure both derivatives with the exact kernel sum and compare with the best-Gaussian value $1 - \sigma + \tfrac78\sigma^2$ of Ref. [2].

In [ ]:
h = 1e-3; Pp = X0.copy(); Pm = X0.copy()
for _ in range(2): Pp = rk4(Pp, h/2); Pm = rk4(Pm, -h/2)
Sigma = (entropy_exact(Pp, sigma0) - entropy_exact(Pm, sigma0)) / (2*h)
dS_dlns = (entropy_exact(X0, sigma0*np.exp(1e-3)) - entropy_exact(X0, sigma0*np.exp(-1e-3))) / 2e-3
print('critical rate of the initial state  R =', round(-Sigma/dS_dlns, 5))
print('best Gaussian state (series)  1 - s + 7/8 s^2 =', round(1 - sigma0 + 7/8*sigma0**2, 5))
print('largest stretching rate  kappa = 1')
for r, t in [(0.9, 0.35), (0.96, 0.5)]:
    k = int(np.argmin(abs(times - t)))
    print(f'check r={r} t={times[k]}:  fft {S[r][k]:.6f}   exact {entropy_exact(states[k], sigma0*np.exp(r*times[k])):.6f}')

## 4. Movie

Top: the coarse-grained density each observer sees (blue dots: the true ensemble, identical in all three; white circle: the blur). Bottom: change of the coarse-grained entropy.

In [ ]:
cols = {rates[0]: '#ff6b6b', rates[1]: '#ffb454', rates[2]: '#5ad1a0'}
bg, fg, mu = '#0b1020', '#e8ecf4', '#9aa3b8'
qlo, qhi, plo, phi, N = np.pi-2.7, np.pi+2.7, -2.4, 2.4, 240
qq = np.linspace(-np.pi, 3*np.pi, 800); sep = 2*np.cos((qq - np.pi)/2)
samp = np.random.default_rng(1).choice(len(weights), size=2500, p=weights)

def image(P, sig):
    h, _, _ = np.histogram2d(P[:, 0], P[:, 1], bins=N, range=[[qlo, qhi], [plo, phi]], weights=weights)
    img = gaussian_filter(h, sig/((qhi-qlo)/N), mode='constant', truncate=4)
    return (img/img.max())**0.5 if img.max() > 0 else img

fig = plt.figure(figsize=(10.8, 10.8), dpi=100, facecolor=bg)
fig.text(0.05, 0.955, 'Seeing the arrow of time through a blurry lens', color=fg, fontsize=25, weight='bold')
fig.text(0.05, 0.925, 'Pendulums balanced on the unstable top, watched through a Gaussian blur σ(t) = %g·e$^{rt}$' % sigma0, color=mu, fontsize=14.5)
ims, dots, circ = [], [], []
for i, r in enumerate(rates):
    ax = fig.add_axes([0.05 + i*0.31, 0.555, 0.28, 0.33]); ax.set_facecolor('#000')
    ims.append(ax.imshow(np.zeros((N, N)).T, origin='lower', extent=[qlo, qhi, plo, phi], cmap='magma', vmin=0, vmax=1, aspect='auto'))
    ax.plot(qq, sep, color='w', lw=0.5, alpha=0.25); ax.plot(qq, -sep, color='w', lw=0.5, alpha=0.25)
    dots.append(ax.plot([], [], '.', color='#9ad7ff', alpha=0.35, ms=1.2)[0])
    ax.set_xlim(qlo, qhi); ax.set_ylim(plo, phi); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_color(cols[r]); sp.set_linewidth(2)
    ax.set_title(('lens fixed   r = 0' if r == 0 else f'blur grows   r = {r:.2f}'), color=cols[r], fontsize=14, pad=8)
    c = plt.Circle((qlo+0.45, plo+0.45), sigma0, fill=False, color='w', lw=1.2); ax.add_patch(c); circ.append(c)
fig.text(0.05, 0.525, 'blue dots: the true ensemble (same in all three)   ·   white circle: size of the blur σ', color=mu, fontsize=12)
a1 = fig.add_axes([0.08, 0.19, 0.50, 0.28], facecolor=bg); a2 = fig.add_axes([0.66, 0.19, 0.30, 0.28], facecolor=bg)
for a in (a1, a2):
    a.tick_params(colors=mu, labelsize=11); a.axhline(0, color='#3a4360', lw=1)
    for sp in a.spines.values(): sp.set_color('#3a4360')
a1.set_xlim(0, T); a1.set_ylim(-1.7, 4.2); a1.set_xlabel('time', color=mu, fontsize=12)
a1.set_ylabel('change of coarse-grained entropy', color=mu, fontsize=12)
a2.set_xlim(0, 1.0); a2.set_ylim(-0.02, 0.04); a2.set_title('first second, zoomed', color=mu, fontsize=12)
lines = {r: (a1.plot([], [], color=cols[r], lw=2.5, label=f'r = {r:g}')[0], a2.plot([], [], color=cols[r], lw=2.5)[0]) for r in rates}
marks = {r: a1.plot([], [], 'o', color=cols[r], ms=7)[0] for r in rates}
a1.legend(loc='upper left', frameon=False, labelcolor=fg, fontsize=12)
fig.text(0.05, 0.105, 'Flow alone (r = 0) makes the coarse-grained entropy fall: the arrow of time runs backwards.\n'
         'The flow stretches at rate κ = 1, but this state sharpens only at R ≈ 0.952 ≈ 1 − σ + ⅞σ²:\n'
         'the finite resolution leaves a margin. Blurring at r = 0.96 is enough; r = 0.90 is not.',
         color=fg, fontsize=13.5, va='top', linespacing=1.5)
fig.text(0.95, 0.012, 'T. Namba · doi:10.5281/zenodo.23320603', color=mu, fontsize=10.5, ha='right')

def update(k):
    t = times[k]
    for i, r in enumerate(rates):
        sig = sigma0*np.exp(r*t); ims[i].set_data(image(states[k], sig).T)
        dots[i].set_data(states[k][samp, 0], states[k][samp, 1]); circ[i].set_radius(min(sig, 3.0))
        dS = S[r][:k+1] - S[r][0]
        lines[r][0].set_data(times[:k+1], dS); lines[r][1].set_data(times[:k+1], dS)
        marks[r].set_data([t], [np.clip(dS[-1], -1.7, 4.2)])
    return []

frames = [0]*10 + list(range(len(times))) + [len(times)-1]*25
movie = FuncAnimation(fig, update, frames=frames, interval=100)
movie.save('arrow_of_time_demo.mp4', fps=10, dpi=100, savefig_kwargs={'facecolor': bg},
           extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p'])
plt.close(fig)
HTML('<video controls loop autoplay muted width="640" src="data:video/mp4;base64,%s"></video>'
     % b64encode(open('arrow_of_time_demo.mp4', 'rb').read()).decode())

## What is proved and what is observed

* **Proved** [1, 2]: the entropy-production identity; $R \le \kappa$ for every Gaussian state in any dimension; the best Gaussian state of the pendulum and its gap $1 - R^* = \sigma - \tfrac78\sigma^2 + \dots$.
* **Observed here**: the critical rate of this ensemble stays below $0.96$ as it evolves (it is no longer Gaussian after $t = 0$), so the blur rate $0.96 < \kappa$ keeps the coarse-grained entropy growing for the whole run.
* **Open**: a proof that a margin proportional to $\sigma$ holds for *every* state, not only Gaussian ones.

The file `arrow_of_time_demo.mp4` appears in the Files panel on the left; right-click → Download.